In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/ml-2/spotify_data.csv")
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,user_id,gender,age,country,subscription_type,listening_time,songs_played_per_day,skip_rate,device_type,ads_listened_per_week,offline_listening,is_churned
0,1.0,Female,54.0,CA,Free,26.0,23.0,0.20,Desktop,31.0,0.0,1
1,2.0,Other,33.0,DE,Family,141.0,62.0,0.34,Web,0.0,1.0,0
2,3.0,Male,38.0,AU,Premium,199.0,38.0,0.04,Mobile,0.0,1.0,1
3,4.0,Female,22.0,CA,Student,36.0,2.0,0.31,Mobile,0.0,1.0,0
4,5.0,Other,29.0,US,Family,250.0,57.0,0.36,Mobile,0.0,1.0,1


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score
from sklearn.linear_model import LogisticRegression

X = df.drop(columns=["is_churned", "user_id"]).dropna()
y = df["is_churned"].loc[X.index]

numberColumns = ["age", "listening_time", "songs_played_per_day", "skip_rate", "ads_listened_per_week", "offline_listening"]
categoryColumns = ["gender", "country", "subscription_type", "device_type"]

X = pd.get_dummies(X, columns=categoryColumns, drop_first=True)
X[numberColumns] = StandardScaler().fit_transform(X[numberColumns])

trainX, testX, trainY, testY = train_test_split(X, y, stratify=y)

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def weightedCost(X, y, w, b, weights):
    m = len(y)
    z = np.dot(X, w) + b
    yHat = sigmoid(z)
    w0, w1 = weights
    return -1 * np.sum( (w1*y*np.log(yHat+1e-9)) + (w0*(1-y)*np.log(1-yHat+1e-9)) ) / m

def gradients(X, y, w, b, weights):
    m = len(y)
    z = np.dot(X, w) + b
    yHat = sigmoid(z)
    w0, w1 = weights
    error = (w1*y-w0*(1-y)) * (yHat-y)
    dw = np.dot(X.T, (yHat-y)) / m
    db = np.sum(yHat-y) / m
    return dw, db

def regression(X, y, lr=0.01, epochs=2000):
    featureCount = X.shape[1]
    w = np.zeros(featureCount)
    b = 0
    pos = (y == 1).sum()
    neg = (y == 0).sum()
    w0 = len(y) / (2*neg)
    w1 = len(y) / (2*pos)
    weights = (w0, w1)
    for i in range(epochs):
        dw, db = gradients(X, y, w, b, weights)
        w -= lr*dw
        b -= lr*db
        if i % 100 == 0:
            cost = weightedCost(X, y, w, b, weights)
            print(f"Epoch {i}, Cost = {cost:.4f}")
    return w, b

NPtrainX = trainX.to_numpy(dtype=float)
nptrainY = trainY.to_numpy(dtype=float)
w, b = regression(NPtrainX, nptrainY)
NPtextX = testX.to_numpy(dtype=float)
prediction = (sigmoid(NPtextX @ w + b) >= 0.5).astype(int)

print("Accuracy:", accuracy_score(testY, prediction))
print("Precision:", precision_score(testY, prediction, zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(testY, prediction))


Epoch 0, Cost = 0.6931
Epoch 100, Cost = 0.7075
Epoch 200, Cost = 0.7320
Epoch 300, Cost = 0.7545
Epoch 400, Cost = 0.7724
Epoch 500, Cost = 0.7859
Epoch 600, Cost = 0.7959
Epoch 700, Cost = 0.8031
Epoch 800, Cost = 0.8083
Epoch 900, Cost = 0.8120
Epoch 1000, Cost = 0.8147
Epoch 1100, Cost = 0.8166
Epoch 1200, Cost = 0.8179
Epoch 1300, Cost = 0.8189
Epoch 1400, Cost = 0.8196
Epoch 1500, Cost = 0.8201
Epoch 1600, Cost = 0.8204
Epoch 1700, Cost = 0.8206
Epoch 1800, Cost = 0.8208
Epoch 1900, Cost = 0.8209
Accuracy: 0.7406471183013145
Precision: 0.0
Confusion Matrix:
 [[1465    0]
 [ 513    0]]


In [ ]:
resampledX, resampledY = RandomOverSampler().fit_resample(trainX, trainY)
NPtrainX2 = resampledX.to_numpy(dtype=float)
NPtrainY2 = resampledY.to_numpy(dtype=float)
w2, b2 = regression(NPtrainX2, NPtrainY2)
prediction2 = (sigmoid(NPtextX @ w2 + b2) >= 0.5).astype(int)

print("After balancing...")
print("Accuracy:", accuracy_score(testY, prediction2))
print("Precision:", precision_score(testY, prediction2, zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(testY, prediction2))

Epoch 0, Cost = 0.6931
Epoch 100, Cost = 0.6928
Epoch 200, Cost = 0.6926
Epoch 300, Cost = 0.6925
Epoch 400, Cost = 0.6924
Epoch 500, Cost = 0.6923
Epoch 600, Cost = 0.6922
Epoch 700, Cost = 0.6921
Epoch 800, Cost = 0.6921
Epoch 900, Cost = 0.6920
Epoch 1000, Cost = 0.6920
Epoch 1100, Cost = 0.6919
Epoch 1200, Cost = 0.6919
Epoch 1300, Cost = 0.6919
Epoch 1400, Cost = 0.6919
Epoch 1500, Cost = 0.6918
Epoch 1600, Cost = 0.6918
Epoch 1700, Cost = 0.6918
Epoch 1800, Cost = 0.6918
Epoch 1900, Cost = 0.6918
After balancing...
Accuracy: 0.5025278058645096
Precision: 0.2559585492227979
Confusion Matrix:
 [[747 718]
 [266 247]]


In [ ]:
model = LogisticRegression(class_weight="balanced", max_iter=2000)
model.fit(trainX, trainY)
predictionSklearn = model.predict(testX)

print("Sklearn...")
print("Accuracy:", accuracy_score(testY, predictionSklearn))
print("Precision:", precision_score(testY, predictionSklearn, zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(testY, predictionSklearn))

Sklearn...
Accuracy: 0.5070778564206269
Precision: 0.27218934911242604
Confusion Matrix:
 [[727 738]
 [237 276]]


در خروجی بخش ب می‌توان دقت بالایی مشاهده کرد، اما اگه دقیق‌ تر بررسی کنیم، می‌ توانیم افزایش هزینه را در مراحل آموزش ببینیم که نشان‌ دهنده‌ ی واگرا شدن مدل است. همینطور اگر به ماتریس  نگاه کنیم، دسته‌ بندی عجیب مدل را خواهیم دید که همه‌ ی داده‌ ها را در یک دسته قرار داده و هیچ داده‌ای در دسته‌ ی دوم وجود ندارد. این مسئله در خروجی صحت نیز مشخص است، که برابر با صفر شده.

اما وقتی به خروجی بخش پ نگاه می‌ کنیم، هم کاهش هزینه در مراحل آموزش را مشاهده می‌کنیم و هم ماتریس  متعادل‌ تری را داریم. دقت این مدل که داده‌های متوازنی داشته از مدل قبلی کمتر است، اما بالا بودن دقت مدل قبلی گمراه کننده است و صحت خروجی مدل بخش پ برابر با صفر نیست. پایین بودن دقت و صحت بخش پ به نسبت انتظاری که از یک مدل قوی داریم را می‌توان به محدودیت‌ های رگرسیون خطی و پیچیدگی داده‌ها نسبت داد، اما در هر صورت برتری داده‌های متوازن به نسبت داده‌های خام نشان داده شده است.

این مسئله در خروجی کتاب‌خانه‌ ی sklearn هم دیده می‌ شود، چون این کتاب‌خانه نیز از روی داده‌های ورودی به دقت و صحتی نزدیک به خروجی بخش پ رسیده و می‌ توان نتیجه گرفت که رگرسیون خطی برای پیش‌ بینی روی این داده‌ها مناسب نیست.
